# Healthcare Operations Analytics — SQL Staging and QA Notebook

## Purpose

This notebook documents the Day 1 SQL staging and quality assurance workflow for the Healthcare Operations Analytics mini project.

The goal is to load CMS hospital data into SQLite, validate source grain and measure coverage, create typed staging views, and export a one-row-per-hospital analytical mart for Day 2 Python analysis.

## Project Scope

This project analyzes emergency-department throughput using three CMS measures:

- EDV — Emergency department volume category
- OP_18b — Median ED arrival-to-departure time for discharged patients
- OP_22 — Percentage of patients who left before being seen

This notebook does not perform benchmarking or visualization. Those tasks begin in Day 2 / Phase 3.

## 1. Project Paths and Environment Setup

This section defines project paths relative to the notebook location. The notebook is expected to run from the `notebooks/` folder.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Notebook is expected to run from the notebooks/ folder
project_root = Path.cwd().parent

raw_data_path = project_root / "data" / "raw"
db_path = project_root / "data" / "database" / "healthcare_operations_cms.db"
processed_path = project_root / "data" / "processed"

print("Project root:", project_root)
print("Raw data path:", raw_data_path)
print("Database path:", db_path)
print("Processed data path:", processed_path)

Project root: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms
Raw data path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\data\raw
Database path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\data\database\healthcare_operations_cms.db
Processed data path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\data\processed


## 2. Raw SQLite Table Creation

This section executes `sql/01_raw_staging.sql`, which creates the raw SQLite tables.

Raw tables are used to preserve the CMS source data before any cleaning, filtering, or transformation.

In [2]:
sql_script_path = project_root / "sql" / "01_raw_staging.sql"

print("SQL script path:", sql_script_path)

# Read SQL script
sql_script = sql_script_path.read_text(encoding="utf-8")

# Create database and execute table creation script
conn = sqlite3.connect(db_path)

try:
    conn.executescript(sql_script)
    conn.commit()
    print("Raw table creation or loading notes.")
finally:
    conn.close()

SQL script path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\sql\01_raw_staging.sql
Raw table creation or loading notes.


In [3]:
import sqlite3
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent
db_path = project_root / "data" / "database" / "healthcare_operations_cms.db"

conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

conn.close()

tables

,name
0,raw_hospital_general
1,raw_timely_effective_care


In [4]:
from pathlib import Path

project_root = Path.cwd().parent
raw_data_path = project_root / "data" / "raw"

print("Raw data folder:")
print(raw_data_path)

raw_files = sorted([p for p in raw_data_path.iterdir() if p.is_file()])

print("\nFiles found:")
for file in raw_files:
    print(f"- {file.name}")

Raw data folder:
C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\data\raw

Files found:
- HOSPITAL_Data_Dictionary.pdf
- Hospital_General_Information.csv
- Timely_and_Effective_Care-Hospital.csv


In [5]:
import pandas as pd
from pathlib import Path

project_root = Path("C:\Projects\AnalyticsPortfolio\healthcare-operations-cms")
raw_data_path = project_root / "data" / "raw"

file_inventory = []

for file in sorted(raw_data_path.iterdir()):
    if file.is_file():
        file_inventory.append({
            "file_name": file.name,
            "extension": file.suffix.lower(),
            "size_kb": round(file.stat().st_size / 1024, 2)
        })

file_inventory_df = pd.DataFrame(file_inventory)
file_inventory_df

,file_name,extension,size_kb
0,HOSPITAL_Data_Dictionary.pdf,.pdf,1261.09
1,Hospital_General_Information.csv,.csv,1419.81
2,Timely_and_Effective_Care-Hospital.csv,.csv,33377.41


## 3. Load Raw CMS Files

This section loads the CMS CSV files into the raw SQLite tables.

Important rules:

- Facility ID is read as text to preserve leading zeroes.
- Raw values are preserved.
- Scores are not cleaned at this stage.
- The load cell should not be rerun repeatedly unless the raw tables are recreated first.

In [6]:
files = {
    "raw_hospital_general": raw_data_path / "Hospital_General_Information.csv",
    "raw_timely_effective_care": raw_data_path / "Timely_and_Effective_Care-Hospital.csv",
}

# Try corrected read settings
raw_hospital_general_preview = pd.read_csv(
    files["raw_hospital_general"],
    encoding="latin1",
    nrows=5
)

raw_timely_effective_care_preview = pd.read_csv(
    files["raw_timely_effective_care"],
    encoding="latin1",
    nrows=5
)

print("=" * 80)
print("raw_hospital_general preview")
print("=" * 80)
display(raw_hospital_general_preview)

print("=" * 80)
print("raw_timely_effective_care preview")
print("=" * 80)
display(raw_timely_effective_care_preview)

raw_hospital_general preview


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,NaN,15,15,NaN,10,10,NaN
1,10005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,NaN,15,15,NaN,10,10,NaN
2,10006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,NaN,15,15,NaN,10,9,NaN
3,10007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,3,2,NaN,15,5,NaN,10,7,NaN
4,10011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,1,5,2,29.0,15,10,29.0,10,7,29.0


raw_timely_effective_care preview


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Condition,Measure ID,Measure Name,Score,Sample,Footnote,Start Date,End Date
0,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Emergency Department,EDV,Emergency department volume,very high,NaN,NaN,01/01/2024,12/31/2024
1,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS,Global Malnutrition Composite Score,Not Available,Not Available,5.0,01/01/2024,12/31/2024
2,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5.0,01/01/2024,12/31/2024
3,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5.0,01/01/2024,12/31/2024
4,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...,Not Available,Not Available,5.0,01/01/2024,12/31/2024


## 4. Raw QA Checks

This section validates the raw source tables before any analytical transformation.

The checks confirm:

- row counts
- unique facility counts
- duplicate keys
- target measure coverage
- reporting windows
- score availability
- join coverage

In [7]:
# Notebook is inside /notebooks, so project root is one level up
project_root = Path.cwd().parent

raw_data_path = project_root / "data" / "raw"
db_path = project_root / "data" / "database" / "healthcare_operations_cms.db"

hospital_csv = raw_data_path / "Hospital_General_Information.csv"
timely_csv = raw_data_path / "Timely_and_Effective_Care-Hospital.csv"

print("Raw data path:", raw_data_path)
print("Database path:", db_path)

# Read raw CSV files as strings to preserve Facility ID, ZIP Code, phone numbers, Not Available, and footnotes
hospital_df = pd.read_csv(
    hospital_csv,
    encoding="latin1",
    dtype="string"
)

timely_df = pd.read_csv(
    timely_csv,
    encoding="latin1",
    dtype="string"
)

# Select and rename only the columns needed for the raw staging tables
hospital_raw = hospital_df[
    [
        "Facility ID",
        "Facility Name",
        "City/Town",
        "State",
        "Hospital Type",
        "Hospital Ownership",
        "Emergency Services",
    ]
].rename(
    columns={
        "Facility ID": "facility_id",
        "Facility Name": "facility_name",
        "City/Town": "city_town",
        "State": "state",
        "Hospital Type": "hospital_type",
        "Hospital Ownership": "hospital_ownership",
        "Emergency Services": "emergency_services",
    }
)

timely_raw = timely_df[
    [
        "Facility ID",
        "Facility Name",
        "Address",
        "City/Town",
        "State",
        "ZIP Code",
        "County/Parish",
        "Telephone Number",
        "Condition",
        "Measure ID",
        "Measure Name",
        "Score",
        "Sample",
        "Footnote",
        "Start Date",
        "End Date",
    ]
].rename(
    columns={
        "Facility ID": "facility_id",
        "Facility Name": "facility_name",
        "Address": "address",
        "City/Town": "city_town",
        "State": "state",
        "ZIP Code": "zip_code",
        "County/Parish": "county_parish",
        "Telephone Number": "telephone_number",
        "Condition": "condition",
        "Measure ID": "measure_id",
        "Measure Name": "measure_name",
        "Score": "score",
        "Sample": "sample",
        "Footnote": "footnote",
        "Start Date": "start_date",
        "End Date": "end_date",
    }
)

# Load into SQLite raw tables
conn = sqlite3.connect(db_path)

try:
    hospital_raw.to_sql(
        "raw_hospital_general",
        conn,
        if_exists="append",
        index=False
    )

    timely_raw.to_sql(
        "raw_timely_effective_care",
        conn,
        if_exists="append",
        index=False
    )

    conn.commit()

    row_counts = pd.read_sql_query(
        """
        SELECT 'raw_hospital_general' AS table_name, COUNT(*) AS row_count
        FROM raw_hospital_general

        UNION ALL

        SELECT 'raw_timely_effective_care' AS table_name, COUNT(*) AS row_count
        FROM raw_timely_effective_care;
        """,
        conn
    )

finally:
    conn.close()

display(row_counts)

Raw data path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\data\raw
Database path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\data\database\healthcare_operations_cms.db


,table_name,row_count
0,raw_hospital_general,5432
1,raw_timely_effective_care,138173


In [8]:
# IMPORTANT:
# This cell appends rows into the raw SQLite tables.
# Run 01_raw_staging.sql first if reloading from scratch.
# Do not rerun this cell repeatedly without recreating the raw tables.

project_root = Path.cwd().parent
db_path = project_root / "data" / "database" / "healthcare_operations_cms.db"

conn = sqlite3.connect(db_path)

try:
    row_counts = pd.read_sql_query(
        """
        SELECT 
            'raw_hospital_general' AS check_name,
            COUNT(*) AS row_count
        FROM raw_hospital_general

        UNION ALL

        SELECT 
            'raw_hospital_general_unique_facilities' AS check_name,
            COUNT(DISTINCT facility_id) AS row_count
        FROM raw_hospital_general

        UNION ALL

        SELECT 
            'raw_timely_effective_care' AS check_name,
            COUNT(*) AS row_count
        FROM raw_timely_effective_care

        UNION ALL

        SELECT 
            'raw_timely_effective_care_unique_facilities' AS check_name,
            COUNT(DISTINCT facility_id) AS row_count
        FROM raw_timely_effective_care;
        """,
        conn
    )

finally:
    conn.close()

display(row_counts)

,check_name,row_count
0,raw_hospital_general,5432
1,raw_hospital_general_unique_facilities,5432
2,raw_timely_effective_care,138173
3,raw_timely_effective_care_unique_facilities,4660


In [9]:
conn = sqlite3.connect(db_path)

try:
    row_counts = pd.read_sql_query(
        """
        SELECT
            facility_id,
            COUNT(*) AS row_count
        FROM raw_hospital_general
        GROUP BY facility_id
        HAVING COUNT(*) > 1;
""",
        conn
    )

finally:
    conn.close()

display(row_counts)

,facility_id,row_count


In [10]:
conn = sqlite3.connect(db_path)

try:
    row_counts = pd.read_sql_query(
        """
        SELECT
            facility_id,
            measure_id,
            COUNT(*) AS row_count
        FROM raw_timely_effective_care
        GROUP BY facility_id, measure_id
        HAVING COUNT(*) > 1;
        """,
        conn
    )

finally:
    conn.close()

display(row_counts)

,facility_id,measure_id,row_count


In [11]:
conn = sqlite3.connect(db_path)
try:
    row_counts = pd.read_sql_query(
        """
       SELECT
            measure_id,
            COUNT(*) AS row_count,
            COUNT(DISTINCT facility_id) AS unique_facilities
        FROM raw_timely_effective_care
        WHERE measure_id IN ('EDV', 'OP_22', 'OP_18b')
        GROUP BY measure_id
        ORDER BY measure_id;
        """,
        conn
    )
finally:
    conn.close()

display(row_counts)

,measure_id,row_count,unique_facilities
0,EDV,4660,4660
1,OP_18b,4660,4660
2,OP_22,4660,4660


In [12]:
conn = sqlite3.connect(db_path)

try:
    row_counts = pd.read_sql_query(
        """
       SELECT
            measure_id,
            start_date,
            end_date,
            COUNT(*) AS row_count
        FROM raw_timely_effective_care
        WHERE measure_id IN ('EDV', 'OP_22', 'OP_18b')
        GROUP BY 
            measure_id,
            start_date,
            end_date
        ORDER BY
            measure_id,
            start_date,
            end_date;
        """,
        conn
    )

finally:
    conn.close()

display(row_counts)

,measure_id,start_date,end_date,row_count
0,EDV,01/01/2024,12/31/2024,4660
1,OP_18b,07/01/2024,06/30/2025,4660
2,OP_22,01/01/2024,12/31/2024,4660


In [13]:
conn = sqlite3.connect(db_path)

try:
    score_availability = pd.read_sql_query(
        """
        SELECT
            measure_id,
            COUNT(*) AS row_count,
            SUM(CASE WHEN score = 'Not Available' THEN 1 ELSE 0 END) AS not_available_count,
            SUM(CASE WHEN score IS NULL OR TRIM(score) = '' THEN 1 ELSE 0 END) AS blank_score_count,
            COUNT(DISTINCT score) AS distinct_score_count
        FROM raw_timely_effective_care
        WHERE measure_id IN ('EDV', 'OP_22', 'OP_18b')
        GROUP BY measure_id
        ORDER BY measure_id;
        """,
        conn
    )

finally:
    conn.close()

display(score_availability)

,measure_id,row_count,not_available_count,blank_score_count,distinct_score_count
0,EDV,4660,823,0,5
1,OP_18b,4660,583,0,277
2,OP_22,4660,828,0,19


In [14]:
conn = sqlite3.connect(db_path)

try:
    row_counts = pd.read_sql_query(
        """
        SELECT
            score AS ed_volume_category,
            COUNT(*) AS row_count
        FROM raw_timely_effective_care
        WHERE measure_id = 'EDV'
        GROUP BY score
        ORDER BY row_count DESC;
        """,
        conn
    )

finally:
    conn.close()

display(row_counts)

,ed_volume_category,row_count
0,low,1666
1,medium,915
2,Not Available,823
3,very high,704
4,high,552


In [15]:
conn = sqlite3.connect(db_path)

try:
    score_patterns = pd.read_sql_query(
        """
        SELECT
            measure_id,
            score,
            COUNT(*) AS row_count
        FROM raw_timely_effective_care
        WHERE measure_id IN ('OP_18b', 'OP_22')
        GROUP BY
            measure_id,
            score
        ORDER BY
            measure_id,
            row_count DESC,
            score;
        """,
        conn
    )

finally:
    conn.close()

display(score_patterns)

,measure_id,score,row_count
0,OP_18b,Not Available,583
1,OP_18b,132,55
2,OP_18b,110,50
3,OP_18b,124,47
4,OP_18b,126,47
...,...,...,...
291,OP_22,13,2
292,OP_22,14,2
293,OP_22,15,1
294,OP_22,16,1


In [16]:
conn = sqlite3.connect(db_path)

try:
    join_coverage = pd.read_sql_query(
        """
        SELECT
            COUNT(DISTINCT te.facility_id) AS target_facilities,
            COUNT(DISTINCT hg.facility_id) AS matched_facilities,
            COUNT(DISTINCT CASE 
                WHEN hg.facility_id IS NULL THEN te.facility_id 
            END) AS missing_from_hospital_general
        FROM raw_timely_effective_care AS te
        LEFT JOIN raw_hospital_general AS hg
            ON te.facility_id = hg.facility_id
        WHERE te.measure_id IN ('EDV', 'OP_22', 'OP_18b');
        """,
        conn
    )

finally:
    conn.close()

display(join_coverage)

,target_facilities,matched_facilities,missing_from_hospital_general
0,4660,4660,0


## 5. Typed Staging View

This section executes `sql/03_staging_views.sql`.

The staging view preserves raw scores while creating safer analysis fields:

- `score_raw`
- `availability_status`
- `score_numeric`
- `score_category`

EDV remains categorical. OP_18b and OP_22 are converted to numeric only when available.

In [17]:
from pathlib import Path

project_root = Path.cwd().parent
sql_script_path = project_root / "sql" / "03_staging_views.sql"

print("SQL script path:", sql_script_path)
print("Exists:", sql_script_path.exists())

print("\nScript preview:")
print(sql_script_path.read_text(encoding="utf-8")[:1500])

SQL script path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\sql\03_staging_views.sql
Exists: True

Script preview:
-- ============================================================
-- Healthcare Operations CMS
-- Block 3: Typed Staging Views
-- Script: 03_staging_views.sql
-- Purpose: Create cleaned measure-level staging views while preserving raw values
-- ============================================================

DROP VIEW IF EXISTS stg_ed_measures;

CREATE VIEW stg_ed_measures AS
SELECT
    facility_id,
    measure_id,
    measure_name,
    score AS score_raw,
    CASE
        WHEN score = 'Not Available' THEN 'Not Available'
        WHEN score IS NULL OR TRIM(score) = '' THEN 'Missing'
        ELSE 'Available'
    END AS availability_status,

    CASE
        WHEN measure_id IN ('OP_18b', 'OP_22')
            AND score <> 'Not Available'
            AND score IS NOT NULL
            AND TRIM(score) <> ''
        THEN CAST(score AS REAL)
        ELSE NULL
    END AS s

In [18]:
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
db_path = project_root / "data" / "database" / "healthcare_operations_cms.db"
sql_script_path = project_root / "sql" / "03_staging_views.sql"

sql_script = sql_script_path.read_text(encoding="utf-8")

conn = sqlite3.connect(db_path)

try:
    conn.executescript(sql_script)
    conn.commit()
    print("03_staging_views.sql executed successfully.")
finally:
    conn.close()

03_staging_views.sql executed successfully.


In [19]:
conn = sqlite3.connect(db_path)

try:
    tables = pd.read_sql_query(
        """
        SELECT name, type
        FROM sqlite_master
        WHERE name = 'stg_ed_measures';
        """,
        conn
    )
finally:
    conn.close()

display(tables)

,name,type
0,stg_ed_measures,view


In [20]:
conn = sqlite3.connect(db_path)

try:
    stg_counts = pd.read_sql_query(
        """
        SELECT
            measure_id,
            COUNT(*) AS row_count,
            SUM(CASE WHEN availability_status = 'Available' THEN 1 ELSE 0 END) AS available_count,
            SUM(CASE WHEN availability_status = 'Not Available' THEN 1 ELSE 0 END) AS not_available_count,
            SUM(CASE WHEN score_numeric IS NOT NULL THEN 1 ELSE 0 END) AS numeric_count,
            SUM(CASE WHEN score_category IS NOT NULL THEN 1 ELSE 0 END) AS category_count
        FROM stg_ed_measures
        GROUP BY measure_id
        ORDER BY measure_id;
        """,
        conn
    )
finally:
    conn.close()

display(stg_counts)

,measure_id,row_count,available_count,not_available_count,numeric_count,category_count
0,EDV,4660,3837,823,0,3837
1,OP_18b,4660,4077,583,4077,0
2,OP_22,4660,3832,828,3832,0


In [21]:
conn = sqlite3.connect(db_path)

try:
    stg_preview = pd.read_sql_query(
        """
        SELECT *
        FROM stg_ed_measures
        LIMIT 10;
        """,
        conn
    )
finally:
    conn.close()

display(stg_preview)

,facility_id,measure_id,measure_name,score_raw,availability_status,score_numeric,score_category,footnote,start_date,end_date
0,010001,EDV,Emergency department volume,very high,Available,NaN,very high,None,01/01/2024,12/31/2024
1,010001,OP_18b,Average (median) time patients spent in the em...,217,Available,217.0,None,None,07/01/2024,06/30/2025
2,010001,OP_22,Left before being seen,3,Available,3.0,None,None,01/01/2024,12/31/2024
3,010005,EDV,Emergency department volume,very high,Available,NaN,very high,None,01/01/2024,12/31/2024
4,010005,OP_18b,Average (median) time patients spent in the em...,141,Available,141.0,None,None,07/01/2024,06/30/2025
5,010005,OP_22,Left before being seen,3,Available,3.0,None,None,01/01/2024,12/31/2024
6,010006,EDV,Emergency department volume,high,Available,NaN,high,None,01/01/2024,12/31/2024
7,010006,OP_18b,Average (median) time patients spent in the em...,144,Available,144.0,None,None,07/01/2024,06/30/2025
8,010006,OP_22,Left before being seen,1,Available,1.0,None,None,01/01/2024,12/31/2024
9,010007,EDV,Emergency department volume,low,Available,NaN,low,None,01/01/2024,12/31/2024


## 6. Hospital-Level Analytical Mart

This section executes `sql/04_analytical_table.sql`.

The resulting view, `mart_hospital_ed_throughput`, has one row per hospital and combines hospital metadata, EDV category, OP_18b, OP_22, availability statuses, footnotes, and reporting dates.

In [22]:
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
db_path = project_root / "data" / "database" / "healthcare_operations_cms.db"
sql_script_path = project_root / "sql" / "04_analytical_table.sql"

sql_script = sql_script_path.read_text(encoding="utf-8")

conn = sqlite3.connect(db_path)

try:
    conn.executescript(sql_script)
    conn.commit()
    print("04_analytical_table.sql executed successfully.")
finally:
    conn.close()

04_analytical_table.sql executed successfully.


In [23]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(db_path)

try:
    views = pd.read_sql_query(
        """
        SELECT name, type
        FROM sqlite_master
        WHERE name = 'mart_hospital_ed_throughput';
        """,
        conn
    )
finally:
    conn.close()

display(views)

,name,type
0,mart_hospital_ed_throughput,view


In [24]:
conn = sqlite3.connect(db_path)

try:
    mart_validation = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS row_count,
            COUNT(DISTINCT facility_id) AS unique_facilities
        FROM mart_hospital_ed_throughput;
        """,
        conn
    )
finally:
    conn.close()

display(mart_validation)

,row_count,unique_facilities
0,4660,4660


In [25]:
conn = sqlite3.connect(db_path)

try:
    mart_duplicates = pd.read_sql_query(
        """
        SELECT
            facility_id,
            COUNT(*) AS row_count
        FROM mart_hospital_ed_throughput
        GROUP BY facility_id
        HAVING COUNT(*) > 1;
        """,
        conn
    )
finally:
    conn.close()

display(mart_duplicates)

,facility_id,row_count


## 7. Export Phase 3 Analysis Input

This section exports the validated analytical mart to:

`data/processed/healthcare_ed_throughput_mart.csv`

This CSV is the approved input for Day 2 Python profiling and peer benchmarking.

In [26]:
import sqlite3
import pandas as pd
from pathlib import Path

# Notebook is inside /notebooks, so project root is one level up
project_root = Path.cwd().parent

db_path = project_root / "data" / "database" / "healthcare_operations_cms.db"
processed_path = project_root / "data" / "processed"
export_path = processed_path / "healthcare_ed_throughput_mart.csv"

# Make sure processed folder exists
processed_path.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(db_path)

try:
    mart_df = pd.read_sql_query(
        """
        SELECT *
        FROM mart_hospital_ed_throughput;
        """,
        conn
    )
finally:
    conn.close()

mart_df.to_csv(export_path, index=False)

print("Export complete.")
print("Rows exported:", len(mart_df))
print("Export path:", export_path)

Export complete.
Rows exported: 4660
Export path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\data\processed\healthcare_ed_throughput_mart.csv


In [27]:
export_check = pd.read_csv(export_path, dtype={"facility_id": "string"})

print("Exported CSV shape:", export_check.shape)
print("Unique facilities:", export_check["facility_id"].nunique())
display(export_check.head())

Exported CSV shape: (4660, 22)
Unique facilities: 4660


,facility_id,facility_name,city_town,state,hospital_type,hospital_ownership,emergency_services,ed_volume_category,op_18b_median_wait_min,op_22_lwbs_pct,...,op_22_availability_status,edv_footnote,op_18b_footnote,op_22_footnote,edv_start_date,edv_end_date,op_18b_start_date,op_18b_end_date,op_22_start_date,op_22_end_date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,very high,217.0,3.0,...,Available,NaN,NaN,NaN,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024
1,010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,very high,141.0,3.0,...,Available,NaN,NaN,NaN,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024
2,010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,Acute Care Hospitals,Proprietary,Yes,high,144.0,1.0,...,Available,NaN,NaN,NaN,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024
3,010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,Acute Care Hospitals,Voluntary non-profit - Private,Yes,low,128.0,1.0,...,Available,NaN,NaN,NaN,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024
4,010011,ST. VINCENT'S EAST,BIRMINGHAM,AL,Acute Care Hospitals,Voluntary non-profit - Private,Yes,NaN,156.0,NaN,...,Not Available,5.0,"3, 29",5,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024


## Phase 2 Completion Checkpoint

Phase 2 is complete.

Completed work:

- Raw CMS files loaded into SQLite.
- Source grain validated.
- Duplicate keys checked.
- Target ED measures confirmed.
- Reporting windows confirmed.
- Score availability checked.
- Join coverage confirmed.
- Typed staging view created.
- Hospital-level analytical mart created.
- Mart exported to `data/processed/healthcare_ed_throughput_mart.csv`.

Locked decisions:

- Facility ID remains text.
- EDV is categorical.
- OP_18b and OP_22 are numeric outcome measures after safe conversion.
- Not Available values are preserved.
- Footnotes and reporting dates are retained.
- Python analysis must use the SQL-validated mart.

Immediate next action:

Begin Day 2 / Phase 3 Python profiling using `healthcare_ed_throughput_mart.csv`.